<a href="https://colab.research.google.com/github/vr11-ai/AiModels/blob/main/Breast_Cancer_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

from google.colab import drive
drive.mount('/content/drive')

dataset_path = '/content/drive/MyDrive/datasets/archive'

print(f"Searching for data in: {dataset_path}")

# --- 3. Locate Metadata and Images ---
metadata_file = None
image_map = {}

print("Mapping file structure...")
for root, dirs, files in os.walk(dataset_path):
    for file in files:

        if file.endswith(".xlsx") and "metadata" in file.lower():
            metadata_file = os.path.join(root, file)

        # Map image filenames to paths
        if file.lower().endswith(('.tif', '.tiff', '.png', '.jpg')):
            image_map[file] = os.path.join(root, file)

if not metadata_file:
    for root, dirs, files in os.walk(dataset_path):
        for file in files:
            if file.endswith(".csv"):
                metadata_file = os.path.join(root, file)
                break

if not metadata_file:
    raise FileNotFoundError("Could not find any Metadata file in your Drive folder.")

print(f"Metadata found: {metadata_file}")
print(f"Total images found: {len(image_map)}")

# --- 4. Load Data & Extract Features ---
if metadata_file.endswith('.xlsx'):
    df = pd.read_excel(metadata_file)
else:
    df = pd.read_csv(metadata_file)

id_col = df.columns[0]
label_col = 'Abnormality' if 'Abnormality' in df.columns else df.columns[-1]

import numpy as np
import cv2
from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops

def extract_composite_features(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None

    img = cv2.resize(img, (128, 128))

    # Apply CLAHE to enhance local contrast (Crucial for medical images)
    #clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    #img = clahe.apply(img)

    # --- HOG ---
    # increased pixels_per_cell to reduce noise in large feature vectors
    hog_feats = hog(img, orientations=9, pixels_per_cell=(8, 8),
                    cells_per_block=(2, 2), block_norm='L2-Hys', visualize=False)

    # --- GLCM ---
    # Added more angles for rotational invariance
    glcm = graycomatrix(img, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                        levels=256, symmetric=True, normed=True)

    contrast = graycoprops(glcm, 'contrast').flatten()
    homogeneity = graycoprops(glcm, 'homogeneity').flatten()
    energy = graycoprops(glcm, 'energy').flatten()
    correlation = graycoprops(glcm, 'correlation').flatten()
    ASM = graycoprops(glcm, 'ASM').flatten() # Added ASM

    glcm_feats = np.hstack([contrast, homogeneity, energy, correlation, ASM])

    # --- LBP ---
    radius = 3
    n_points = 8 * radius
    lbp = local_binary_pattern(img, n_points, radius, method='uniform')
    (lbp_hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, n_points + 3), range=(0, n_points + 2))
    lbp_hist = lbp_hist.astype("float")
    lbp_hist /= (lbp_hist.sum() + 1e-7)

    return np.hstack([hog_feats, glcm_feats, lbp_hist])

    # Calculate histogram of LBP codes to get a feature vector
    (lbp_hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, n_points + 3), range=(0, n_points + 2))

    # Normalize the histogram
    lbp_hist = lbp_hist.astype("float")
    lbp_hist /= (lbp_hist.sum() + 1e-7)

    # --- FUSION: Combine all features ---
    # Concatenate all 1D arrays into one long feature vector
    combined_features = np.hstack([hog_feats, glcm_feats, lbp_hist])

    return combined_features

X_data, y_labels = [], []
print("Extracting features...")

for idx, row in df.iterrows():
    img_id = str(row[id_col])
    full_path = None
    # Check for various extensions in your Archive folder
    for ext in ['.tif', '.tiff', '.png', '.jpg', '']:
        test_name = img_id + ext
        if test_name in image_map:
            full_path = image_map[test_name]
            break

    if full_path:
        feats = extract_composite_features(full_path)
        if feats is not None:
            X_data.append(feats)
            y_labels.append(str(row[label_col]))

X = np.array(X_data)
y = np.array(y_labels)
print(f"Successfully processed {len(X)} images.")

# --- 5. Train & Evaluate ---
if len(X) > 0:
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.3,random_state=42)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)


    clf = SVC(kernel='rbf', probability=True, class_weight='balanced')
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    unique_labels = np.unique(np.concatenate((y_test, y_pred)))
    filtered_names = [str(le.classes_[i]) for i in unique_labels]

    print("\nAccuracy: {:.2f}%".format(accuracy_score(y_test, y_pred) * 100))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Searching for data in: /content/drive/MyDrive/datasets/archive
Mapping file structure...
Metadata found: /content/drive/MyDrive/datasets/archive/Metadata.xlsx
Total images found: 553
Extracting features...
Successfully processed 22 images.

Accuracy: 42.86%


In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import VGG19
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from PIL import Image
import cv2 # Added OpenCV import for image conversion

from google.colab import drive
drive.mount('/content/drive')

# ----------------------------
# CONFIG
# ----------------------------
META_DIR = "/content/drive/MyDrive/datasets/archive"
ORIGINAL_IMG_DIR = "/content/drive/MyDrive/datasets/archive"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15
MIN_SAMPLES_PER_CLASS = 3

print("TensorFlow Version:", tf.__version__)

# FIND METADATA FILE
meta_files = []
for root, dirs, files in os.walk(META_DIR):
    for f in files:
        if f.endswith(('.csv', '.xlsx')):
            meta_files.append(os.path.join(root, f))

if not meta_files:
    raise FileNotFoundError("No metadata file (.csv/.xlsx) found!")

meta_path = meta_files[0]
print("Metadata Loaded:", meta_path)

# LOAD METADATA
if meta_path.endswith('.xlsx'):
    df = pd.read_excel(meta_path)
else:
    df = pd.read_csv(meta_path)

df.columns = df.columns.astype(str).str.strip()
print("\nColumns Found:", df.columns.tolist())

# AUTO DETECT LABEL COLUMN
TARGET_COLUMN_NAME = None

for col in df.columns:
    unique_vals = df[col].astype(str).str.upper().unique()
    if len(unique_vals) > 1 and len(unique_vals) < 20:
        TARGET_COLUMN_NAME = col
        break

if TARGET_COLUMN_NAME is None:
    raise ValueError("Could not detect label column automatically!")

print("\nDetected Label Column:", TARGET_COLUMN_NAME)

# ID COLUMN (FIRST COLUMN)
id_col = df.columns[0]
print("Using ID Column:", id_col)

# CLEAN LABELS
df[TARGET_COLUMN_NAME] = df[TARGET_COLUMN_NAME].astype(str).str.upper().str.strip()
df['final_label'] = df[TARGET_COLUMN_NAME]

df = df[df['final_label'].notna()]
df = df[df['final_label'] != "NAN"]

print("\nClass Counts:")
print(df['final_label'].value_counts())

# NORMALIZE FUNCTION FOR MATCHING
def normalize_name(x):
    x = str(x).strip()
    x = x.replace(".0", "")
    x = x.replace(" ", "")
    x = x.replace("_", "")
    x = x.lower()
    return x

# IMAGE CONVERSION TO PNG (for Pillow compatibility)
CONVERTED_IMG_DIR = os.path.join(ORIGINAL_IMG_DIR, 'converted_for_keras')
os.makedirs(CONVERTED_IMG_DIR, exist_ok=True)

print(f"\nChecking for converted images in: {CONVERTED_IMG_DIR}")

# Check if converted images already exist to skip re-conversion
if not os.listdir(CONVERTED_IMG_DIR) or not any(f.lower().endswith(('.png', '.jpg', '.jpeg')) for f in os.listdir(CONVERTED_IMG_DIR)):
    print("Converted images not found or directory empty. Starting conversion...")

    original_image_paths = []
    # Collect all original TIFF image paths from ORIGINAL_IMG_DIR
    for root, dirs, files in os.walk(ORIGINAL_IMG_DIR):
        for f in files:
            if f.lower().endswith(('.tif', '.tiff')):
                original_image_paths.append(os.path.join(root, f))

    if not original_image_paths:
        print("No TIFF images found for conversion.")
    else:
        for original_path in original_image_paths:
            img_name_no_ext = os.path.splitext(os.path.basename(original_path))[0]
            converted_path = os.path.join(CONVERTED_IMG_DIR, f"{img_name_no_ext}.png")

            try:
                img = cv2.imread(original_path) # Use OpenCV to read original TIFF
                if img is None:
                    print(f"Warning: Could not read {original_path} with OpenCV. Skipping.")
                    continue

                # Ensure 3 channels for VGG16, convert grayscale to BGR if necessary
                if len(img.shape) == 2: # Grayscale image
                    img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
                elif img.shape[2] == 4: # RGBA image
                    img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)

                cv2.imwrite(converted_path, img) # Save as PNG
            except Exception as e:
                print(f"Error converting {original_path} to PNG: {e}")
        print(f"Conversion complete. Converted {len(original_image_paths)} images to PNG in {CONVERTED_IMG_DIR}")
else:
    print("Converted images already exist. Skipping conversion.")

# Update IMG_DIR to point to the converted images for subsequent steps
IMG_DIR = CONVERTED_IMG_DIR

# INDEX IMAGE FILES (from converted directory)
print("\nIndexing image files...")

file_map = {}
for root, dirs, files in os.walk(IMG_DIR):
    for f in files:
        # Now only look for PNG, JPG, JPEG as TIFFs should be converted
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            name_no_ext = os.path.splitext(f)[0]
            file_map[normalize_name(name_no_ext)] = os.path.join(root, f)

print("Total Images Indexed:", len(file_map))

# MATCH IMAGE PATHS
df['norm_id'] = df[id_col].apply(normalize_name)
df['filepath'] = df['norm_id'].map(file_map)

df = df.dropna(subset=['filepath'])

print("\nImages Successfully Matched:", len(df))

if len(df) == 0:
    print("\nSample Metadata IDs:", df[id_col].head(10).tolist())
    print("\nSample Image Filenames:", list(file_map.keys())[:10])
    raise ValueError("No images matched with metadata IDs. IDs and filenames are different.")

# REMOVE SMALL CLASSES
df = df[df['final_label'].map(df['final_label'].value_counts()) >= MIN_SAMPLES_PER_CLASS]

print("\nClass Counts After Filtering Small Classes:")
print(df['final_label'].value_counts())

if len(df) == 0:
    raise ValueError("No data left after filtering small classes. Reduce MIN_SAMPLES_PER_CLASS.")

# TRAIN / VALID SPLIT
train_df, valid_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['final_label'],
    random_state=42
)

print("\nTrain size:", len(train_df))
print("Validation size:", len(valid_df))

# GENERATORS
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2]
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_dataframe(
    train_df,
    x_col='filepath',
    y_col='final_label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_gen = val_datagen.flow_from_dataframe(
    valid_df,
    x_col='filepath',
    y_col='final_label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# MODEL
base_model = VGG19(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

for layer in base_model.layers:
    layer.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(len(train_gen.class_indices), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# TRAIN
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS
)

print("\nTraining Finished Successfully!")

# EVALUATE
val_loss, val_acc = model.evaluate(val_gen)
print("\nValidation Accuracy:", val_acc * 100, "%")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TensorFlow Version: 2.19.0
Metadata Loaded: /content/drive/MyDrive/datasets/archive/Metadata.xlsx

Columns Found: ['1st column:', 'Image reference number.', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7']

Detected Label Column: Unnamed: 2
Using ID Column: 1st column:

Class Counts:
final_label
G    548
F     78
D     19
-      2
Name: count, dtype: int64

Checking for converted images in: /content/drive/MyDrive/datasets/archive/converted_for_keras
Converted images already exist. Skipping conversion.

Indexing image files...
Total Images Indexed: 42

Images Successfully Matched: 61

Class Counts After Filtering Small Classes:
final_label
G    52
F     5
D     4
Name: count, dtype: int64

Train size: 48
Validation size: 13
Found 48 validated image filenames belonging to 3 classes.
Found 13 validated image filenames belonging

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv4 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv4 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv4 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 20,156,483 (76.89 MB)

 Trainable params: 132,099 (516.01 KB)

 Non-trainable params: 20,024,384 (76.39 MB)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 62s 42s/step - accuracy: 0.6042 - loss: 0.9176 - val_accuracy: 0.8462 - val_loss: 0.7327
Epoch 2/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 26s 18s/step - accuracy: 0.6875 - loss: 0.8554 - val_accuracy: 0.8462 - val_loss: 0.6876
Epoch 3/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 25s 18s/step - accuracy: 0.6528 - loss: 0.8105 - val_accuracy: 0.8462 - val_loss: 0.6507
Epoch 4/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 25s 18s/step - accuracy: 0.7292 - loss: 0.8008 - val_accuracy: 0.8462 - val_loss: 0.6221
Epoch 5/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 25s 19s/step - accuracy: 0.8472 - loss: 0.6090 - val_accuracy: 0.8462 - val_loss: 0.6003
Epoch 6/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 24s 11s/step - accuracy: 0.7917 - loss: 0.7686 - val_accuracy: 0.8462 - val_loss: 0.5839
Epoch 7/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 23s 11s/step - accuracy: 0.8507 - loss: 0.5831 - val_accuracy: 0.8462 - val_loss: 0.5715
Epoch 8/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 28s 11s/step - accuracy: 0.7986 - loss: 0.6490 - val_accuracy: 0.8462 - val_loss: 0.5620


In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import cv2

from google.colab import drive
drive.mount('/content/drive')

from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split

ORIGINAL_DATA_DIR = '/content/drive/MyDrive/datasets/archive'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15

TARGET_COLUMN_NAME = 'Unnamed: 2'

print("Original Dataset Path:", ORIGINAL_DATA_DIR)

meta_files = []
for root, dirs, files in os.walk(ORIGINAL_DATA_DIR):
    for f in files:
        if f.endswith(('.csv', '.xlsx')):
            meta_files.append(os.path.join(root, f))

if not meta_files:
    raise FileNotFoundError("No CSV/Excel found inside dataset folder!")

meta_path = meta_files[0]
print("Metadata File Loaded:\n", meta_path)

if meta_path.endswith('.xlsx'):
    df = pd.read_excel(meta_path)
else:
    df = pd.read_csv(meta_path)

print("\nColumns Found:", df.columns.tolist())

if TARGET_COLUMN_NAME not in df.columns:
    raise ValueError(f"Column '{TARGET_COLUMN_NAME}' not found. Available columns: {df.columns.tolist()}")

# First, remove rows where the target label is NaN or empty
initial_len = len(df)
df[TARGET_COLUMN_NAME] = df[TARGET_COLUMN_NAME].astype(str).str.strip()
df = df[df[TARGET_COLUMN_NAME] != 'nan']
df = df[df[TARGET_COLUMN_NAME] != 'None']
df = df[df[TARGET_COLUMN_NAME] != '']
print(f"Filtered out {initial_len - len(df)} rows with 'nan' or empty labels.")

df['final_label'] = df[TARGET_COLUMN_NAME]

print("\nFinal Class Counts:\n", df['final_label'].value_counts())

if len(df) == 0:
    raise ValueError("No valid rows found after filtering. Check your data and TARGET_COLUMN_NAME.")

CONVERTED_IMG_DIR = os.path.join(ORIGINAL_DATA_DIR, 'converted_for_keras')
os.makedirs(CONVERTED_IMG_DIR, exist_ok=True)

print(f"\nChecking for converted images in: {CONVERTED_IMG_DIR}")

if not os.listdir(CONVERTED_IMG_DIR) or not any(f.lower().endswith(('.png', '.jpg', '.jpeg')) for f in os.listdir(CONVERTED_IMG_DIR)):
    print("Converted images not found or directory empty. Starting conversion...")

    original_image_paths = []
    # Collect all original TIFF image paths from ORIGINAL_DATA_DIR
    for root, dirs, files in os.walk(ORIGINAL_DATA_DIR):
        for f in files:
            if f.lower().endswith(('.tif', '.tiff')):
                original_image_paths.append(os.path.join(root, f))

    if not original_image_paths:
        print("No TIFF images found for conversion.")
    else:
        for original_path in original_image_paths:
            img_name_no_ext = os.path.splitext(os.path.basename(original_path))[0]
            converted_path = os.path.join(CONVERTED_IMG_DIR, f"{img_name_no_ext}.png")

            try:
                img = cv2.imread(original_path) # Use OpenCV to read original TIFF
                if img is None:
                    print(f"Warning: Could not read {original_path} with OpenCV. Skipping.")
                    continue

                # Ensure 3 channels for VGG16, convert grayscale to BGR if necessary
                if len(img.shape) == 2: # Grayscale image
                    img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
                elif img.shape[2] == 4: # RGBA image
                    img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)

                cv2.imwrite(converted_path, img)
            except Exception as e:
                print(f"Error converting {original_path} to PNG: {e}")
        print(f"Conversion complete. Converted {len(original_image_paths)} images to PNG in {CONVERTED_IMG_DIR}")
else:
    print("Converted images already exist. Skipping conversion.")

# Update DATA_DIR to point to the converted images for subsequent steps
DATA_DIR = CONVERTED_IMG_DIR

# File Path Matching (using converted images)
print("\nIndexing image files...")

file_map = {}
for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            name_no_ext = os.path.splitext(f)[0]
            file_map[name_no_ext] = os.path.join(root, f)

print("Total Image Files Indexed:", len(file_map))

id_col = df.columns[0]

def get_path(img_id):
    img_id = str(img_id).strip()
    return file_map.get(img_id, None)

df['filepath'] = df[id_col].apply(get_path)

df = df.dropna(subset=['filepath'])
print("Images Successfully Matched:", len(df))

if len(df) == 0:
    raise ValueError("No images matched with metadata IDs. Check file names and ID column.")

# Train/Validation Split
# Ensure there are enough samples per class for stratification
class_counts = df['final_label'].value_counts()
valid_classes_for_stratify = class_counts[class_counts >= 2].index.tolist()
df_stratified = df[df['final_label'].isin(valid_classes_for_stratify)]

if len(df_stratified) == 0:
    print("Warning: Not enough classes with 2+ samples for stratified split. Proceeding with simple split.")
    train_df, valid_df = train_test_split(
        df,
        test_size=0.2,
        random_state=42
    )
else:
    train_df, valid_df = train_test_split(
        df_stratified,
        test_size=0.2,
        stratify=df_stratified['final_label'],
        random_state=42
    )


print("\nTrain size:", len(train_df))
print("Validation size:", len(valid_df))

# Data Generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2]
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_dataframe(
    train_df,
    x_col='filepath',
    y_col='final_label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_gen = val_datagen.flow_from_dataframe(
    valid_df,
    x_col='filepath',
    y_col='final_label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# 8. Build Model (VGG16)
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

for layer in base_model.layers:
    layer.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(len(train_gen.class_indices), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# Train Model
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS
)

print("\nTraining Finished Successfully!")

# Evaluate Model
val_loss, val_acc = model.evaluate(val_gen)
print(f"\nValidation Accuracy: {val_acc * 100:.2f}%")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TensorFlow Version: 2.19.0
Original Dataset Path: /content/drive/MyDrive/datasets/archive
Metadata File Loaded:
 /content/drive/MyDrive/datasets/archive/Metadata.xlsx

Columns Found: ['1st column:', ' Image reference number.', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7']
Filtered out 30 rows with 'nan' or empty labels.

Final Class Counts:
 final_label
G    548
F     78
D     19
-      2
Name: count, dtype: int64

Checking for converted images in: /content/drive/MyDrive/datasets/archive/converted_for_keras
Converted images already exist. Skipping conversion.

Indexing image files...
Total Image Files Indexed: 42
Images Successfully Matched: 61

Train size: 48
Validation size: 13
Found 48 validated image filenames belonging to 3 classes.
Found 13 validated image filenames belonging to 3 classes.


Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_6      │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,846,787 (56.64 MB)

 Trainable params: 132,099 (516.01 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 68s 46s/step - accuracy: 0.5347 - loss: 0.9812 - val_accuracy: 0.8462 - val_loss: 0.8452
Epoch 2/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 60s 38s/step - accuracy: 0.5000 - loss: 0.9596 - val_accuracy: 0.8462 - val_loss: 0.7825
Epoch 3/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 59s 37s/step - accuracy: 0.6389 - loss: 0.8709 - val_accuracy: 0.8462 - val_loss: 0.7283
Epoch 4/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 58s 37s/step - accuracy: 0.7500 - loss: 0.6983 - val_accuracy: 0.8462 - val_loss: 0.6821
Epoch 5/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 58s 38s/step - accuracy: 0.6667 - loss: 0.8309 - val_accuracy: 0.8462 - val_loss: 0.6440
Epoch 6/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 59s 20s/step - accuracy: 0.8715 - loss: 0.6339 - val_accuracy: 0.8462 - val_loss: 0.6135
Epoch 7/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 59s 22s/step - accuracy: 0.7986 - loss: 0.7059 - val_accuracy: 0.8462 - val_loss: 0.5903
Epoch 8/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 60s 22s/step - accuracy: 0.8368 - loss: 0.6479 - val_accuracy: 0.8462 - val_loss: 0.5723
